In [ ]:
!pip install biopython --quiet

import os
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from Bio import SeqIO
from Bio.Align import PairwiseAligner

In [ ]:
DATASET_DIR = 'Dataset'
FILE_SARS2 = os.path.join(DATASET_DIR, 'sars_cov_2_spike.fasta')
FILE_SARS = os.path.join(DATASET_DIR, 'sars_cov_spike.fasta')

URL_SARS2 = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_045512.2&rettype=fasta&retmode=text'
URL_SARS = 'https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi?db=nuccore&id=NC_004718.3&rettype=fasta&retmode=text'

SPIKE_SLICE = {
    'sars2': (21563, 25384),
    'sars': (21492, 25259),
}

os.makedirs(DATASET_DIR, exist_ok=True)

if not os.path.exists(FILE_SARS2):
    print(f'Downloading {URL_SARS2} -> {FILE_SARS2}')
    urllib.request.urlretrieve(URL_SARS2, FILE_SARS2)

if not os.path.exists(FILE_SARS):
    print(f'Downloading {URL_SARS} -> {FILE_SARS}')
    urllib.request.urlretrieve(URL_SARS, FILE_SARS)

In [ ]:
def load_spike(path, slice_key):
    record = next(SeqIO.parse(path, 'fasta'))
    seq = str(record.seq).upper()
    
    if len(seq) > 10_000:
        start, stop = SPIKE_SLICE[slice_key]
        seq = seq[start - 1:stop]
        
    return seq, record.id, record.description

In [ ]:
spike_sars2, id_sars2, desc_sars2 = load_spike(FILE_SARS2, 'sars2')
spike_sars, id_sars, desc_sars = load_spike(FILE_SARS, 'sars')

print(f'SARS-CoV-2 ({id_sars2}): {len(spike_sars2)} bp')
print(f'SARS-CoV ({id_sars}): {len(spike_sars)} bp')

In [ ]:
def clean_sequence(seq):
    raw_len = len(seq)
    raw_lower = 0
    
    for base in seq:
        if base.islower():
            raw_lower += 1
    
    raw_gaps = seq.count('-')
    
    cleaned = seq.upper().replace('-', '')
    
    valid = set('ATGC')
    ambiguous = {}
    
    for base in set(cleaned):
        if base not in valid:
            ambiguous[base] = cleaned.count(base)
    
    return cleaned, {
        'Raw Length': raw_len,
        'Cleaned Length': len(cleaned),
        'Lowercased Letters': raw_lower,
        'Erased Gaps': raw_gaps,
        'Ambigous Bases': ambiguous or '-',
    }

In [ ]:
spike_sars2_clean, audit_sars2 = clean_sequence(spike_sars2)
spike_sars_clean, audit_sars = clean_sequence(spike_sars)

audit_df = pd.DataFrame({'SARS-CoV-2': audit_sars2, 'SARS-CoV': audit_sars})
audit_df

In [ ]:
def base_counts(seq):
    counts = {}
    
    for base in 'ATGC':
        counts[base] = seq.count(base)
        
    return counts

In [ ]:
comp_df = pd.DataFrame({
    'SARS-CoV-2': base_counts(spike_sars2_clean),
    'SARS-CoV': base_counts(spike_sars_clean),
})

ax = comp_df.plot(kind='bar', figsize=(8, 4), color=['#1f77b4', '#ff7f0e'])
ax.set_title('Nucleotide Composition of Spike Gene - SARS-CoV-2 vs SARS-CoV')
ax.set_xlabel('Nucleotide base')
ax.set_ylabel('Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

comp_df

In [ ]:
def gc_windows(seq, window, step):
    gc = []
    
    for i in range(0, len(seq) - window + 1, step):
        sub = seq[i:i + window]
        gc.append((sub.count('G') + sub.count('C')) / window * 100)
        
    return np.array(gc)

In [ ]:
WINDOW = 60
STEP = 30

gc_sars2 = gc_windows(spike_sars2_clean, WINDOW, STEP)
gc_sars = gc_windows(spike_sars_clean, WINDOW, STEP)

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(gc_sars2, bins=20, alpha=0.6, label='SARS-CoV-2', color='#1f77b4')
ax.hist(gc_sars, bins=20, alpha=0.6, label='SARS-CoV', color='#ff7f0e')
ax.set_title(f'GC% Distribution (Sliding Window {WINDOW} bp, step {STEP})')
ax.set_xlabel('GC content (%)')
ax.set_ylabel('Window count')
ax.legend()
plt.tight_layout()
plt.show()

print(f'GC% SARS-CoV-2 : mean={gc_sars2.mean():.2f}, std={gc_sars2.std():.2f}')
print(f'GC% SARS-CoV : mean={gc_sars.mean():.2f}, std={gc_sars.std():.2f}')

In [ ]:
min_len = min(len(gc_sars2), len(gc_sars))
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(gc_sars2[:min_len], gc_sars[:min_len], alpha=0.5, s=20)
lims = [25, 60]
ax.plot(lims, lims, 'r--', label='y = x')
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel('GC% window SARS-CoV-2')
ax.set_ylabel('GC% window SARS-CoV')
ax.set_title('GC% Correlation per Window')
ax.legend()
plt.tight_layout()
plt.show()

r = np.corrcoef(gc_sars2[:min_len], gc_sars[:min_len])[0, 1]
print(f'Pearson Correlation GC% window: r = {r:.3f}')

In [ ]:
def needleman_wunsch(seq1, seq2, match, mismatch, gap):
    n = len(seq1)
    m = len(seq2)
    
    dp = np.zeros((n + 1, m + 1), dtype=np.int32)
    
    dp[:, 0] = np.arange(n + 1) * gap
    dp[0, :] = np.arange(m + 1) * gap
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if seq1[i - 1] == seq2[j - 1]:
                subtitution_score = match
            else:
                subtitution_score = mismatch

            dp[i, j] = max(
                dp[i - 1, j - 1] + subtitution_score,
                dp[i - 1, j] + gap,
                dp[i, j - 1] + gap,
            )
            
    aligned_seq1 = []
    aligned_seq2 = []

    row = n
    col = m

    while row > 0 or col > 0:
        if row > 0 and col > 0:
            if seq1[row - 1] == seq2[col - 1]:
                subtitution_score = match
            else:
                subtitution_score = mismatch
            
            if dp[row, col] == dp[row - 1, col - 1] + subtitution_score:
                aligned_seq1.append(seq1[row - 1])
                aligned_seq2.append(seq2[col - 1])
                row -= 1
                col -= 1
                continue
        
        if row > 0 and dp[row, col] == dp[row - 1, col] + gap:
            aligned_seq1.append(seq1[row - 1])
            aligned_seq2.append('-')
            row -= 1
        else:
            aligned_seq1.append('-')
            aligned_seq2.append(seq2[col - 1])
            col -= 1
            
    return ''.join(reversed(aligned_seq1)), ''.join(reversed(aligned_seq2)), int(dp[n, m]), dp

In [ ]:
def smith_waterman(seq1, seq2, match, mismatch, gap):
    n = len(seq1)
    m = len(seq2)
    
    dp = np.zeros((n + 1, m + 1), dtype=np.int32)
    
    max_score = 0
    max_pos = (0, 0)
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if seq1[i - 1] == seq2[j - 1]:
                subtitution_score = match
            else:
                subtitution_score = mismatch

            dp[i, j] = max(
                0,
                dp[i - 1, j - 1] + subtitution_score,
                dp[i - 1, j] + gap,
                dp[i, j - 1] + gap,
            )
            
            if dp[i, j] > max_score:
                max_score = dp[i, j]
                max_pos = (i, j)
            
    aligned_seq1 = []
    aligned_seq2 = []

    row = max_pos[0]
    col = max_pos[1]

    while row > 0 and col > 0 and dp[row, col] > 0:
        if seq1[row - 1] == seq2[col - 1]:
            subtitution_score = match
        else:
            subtitution_score = mismatch
        
        if dp[row, col] == dp[row - 1, col - 1] + subtitution_score:
            aligned_seq1.append(seq1[row - 1])
            aligned_seq2.append(seq2[col - 1])
            row -= 1
            col -= 1
        elif dp[row, col] == dp[row - 1, col] + gap:
            aligned_seq1.append(seq1[row - 1])
            aligned_seq2.append('-')
            row -= 1
        else:
            aligned_seq1.append('-')
            aligned_seq2.append(seq2[col - 1])
            col -= 1
            
    return ''.join(reversed(aligned_seq1)), ''.join(reversed(aligned_seq2)), int(max_score), dp

In [ ]:
def slice_by_residues(seq, start_residue, end_residue):
    dna_start = (start_residue - 1) * 3 + 1
    dna_end = end_residue * 3
    
    return seq[dna_start - 1:dna_end]


In [ ]:
RBD_START_RESIDUE = 319
RBD_END_RESIDUE = 541

rbd_sars2 = slice_by_residues(spike_sars2_clean, RBD_START_RESIDUE, RBD_END_RESIDUE)
rbd_sars = slice_by_residues(spike_sars_clean, RBD_START_RESIDUE, RBD_END_RESIDUE)

print(f'RBD SARS-CoV-2: {len(rbd_sars2)} bp')
print(f'RBD SARS-CoV : {len(rbd_sars)} bp')

In [ ]:
MATCH = 2
MISMATCH = -1
GAP = -2

nw_aligned_sars2, nw_aligned_sars, nw_score, nw_dp_matrix = needleman_wunsch(
                                                                rbd_sars2, 
                                                                rbd_sars, 
                                                                MATCH, 
                                                                MISMATCH, 
                                                                GAP
                                                            )

print(f'Needleman-Wunsch (NW) score = {nw_score}')

sw_aligned_sars2, sw_aligned_sars, sw_score, sw_dp_matrix = smith_waterman(
                                                                rbd_sars2, 
                                                                rbd_sars, 
                                                                MATCH, 
                                                                MISMATCH, 
                                                                GAP
                                                            )

print(f'Smith-Waterman (SW) score = {sw_score}')

In [ ]:
def make_aligner(mode):
    aligner = PairwiseAligner()
    aligner.mode = mode
    aligner.match_score = MATCH
    aligner.mismatch_score = MISMATCH
    aligner.open_gap_score = GAP
    aligner.extend_gap_score = GAP
    
    return aligner

In [ ]:
global_aligner = make_aligner('global')
local_aligner = make_aligner('local')

bio_nw_score = global_aligner.score(rbd_sars2, rbd_sars)
bio_sw_score = local_aligner.score(rbd_sars2, rbd_sars)

print(f'Custom Needleman-Wunsch (Global) = {nw_score}, Biopython global = {bio_nw_score:.0f}')
print(f'Custom Smith-Waterman (Local) = {sw_score}, Biopython local = {bio_sw_score:.0f}')

if nw_score != int(bio_nw_score):
    print(f'Needleman-Wunsch mismatch: custom={nw_score}, biopython={int(bio_nw_score)}')
elif sw_score != int(bio_sw_score):
    print(f'Smith-Waterman mismatch: custom={sw_score}, biopython={int(bio_sw_score)}')
else:
    print('custom implementation matches Biopython.')

In [ ]:
full_nw_score = global_aligner.score(spike_sars2_clean, spike_sars_clean)
full_sw_score = local_aligner.score(spike_sars2_clean, spike_sars_clean)

full_nw_aln = global_aligner.align(spike_sars2_clean, spike_sars_clean)[0]
full_sw_aln = local_aligner.align(spike_sars2_clean, spike_sars_clean)[0]

print(f'Full Spike Needleman-Wunsch score = {full_nw_score:.0f}')
print(f'Full Spike Smith-Waterman score = {full_sw_score:.0f}')

In [ ]:
def alignment_stats(aligned_seq1, aligned_seq2):
    matches = 0
    mismatches = 0
    gaps = 0

    for base1, base2 in zip(aligned_seq1, aligned_seq2):
        if base1 == '-' or base2 == '-':
            gaps += 1
        elif base1 == base2:
            matches += 1
        else:
            mismatches += 1

    alignment_length = len(aligned_seq1)

    return {
        'Alignment Length': alignment_length,
        'Matches': matches,
        'Mismatches': mismatches,
        'Gaps': gaps,
        'Percent Identity': round(matches / alignment_length * 100, 2),
    }

In [ ]:
stats_nw = alignment_stats(nw_aligned_sars2, nw_aligned_sars)
stats_sw = alignment_stats(sw_aligned_sars2, sw_aligned_sars)

results_df = pd.DataFrame({
    'NW (RBD, custom)': {**stats_nw, 'score': nw_score},
    'SW (RBD, custom)': {**stats_sw, 'score': sw_score},
    'NW (Full Spike, Biopy)': {'score': int(full_nw_score)},
    'SW (Full Spike, Biopy)': {'score': int(full_sw_score)},
})

results_df

In [ ]:
def display_alignment(aligned_seq1, aligned_seq2, seq1_label, seq2_label, length=60, max_cols=120):
    ruler_chars = []
    
    for base1, base2 in zip(aligned_seq1, aligned_seq2):
        if base1 == base2 and base1 != '-':
            ruler_chars.append('|')
        elif '-' in (base1, base2):
            ruler_chars.append(' ')
        else:
            ruler_chars.append('.')
            
    ruler = ''.join(ruler_chars)

    output_lines = []
    
    for chunk_start in range(0, min(len(aligned_seq1), max_cols), length):
        chunk_end = chunk_start + length
        output_lines.append(f'{seq1_label:>5} {chunk_start + 1:>5}  {aligned_seq1[chunk_start:chunk_end]}')
        output_lines.append(f'{"":>12} {ruler[chunk_start:chunk_end]}')
        output_lines.append(f'{seq2_label:>5} {chunk_start + 1:>5}  {aligned_seq2[chunk_start:chunk_end]}')
        output_lines.append('')
        
    return '\n'.join(output_lines)

In [ ]:
def print_title(title, length):
    print(f' {title} '.center(length, '='))

In [ ]:
print_title('Needleman-Wunsch (RBD)', 73)
print(display_alignment(nw_aligned_sars2, nw_aligned_sars, 'SARS2', 'SARS'))

print_title('Smith-Waterman (RBD)', 73)
print(display_alignment(sw_aligned_sars2, sw_aligned_sars, 'SARS2', 'SARS'))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(sw_dp_matrix, cmap='viridis', aspect='auto')
ax.set_title('Smith-Waterman DP Matrix (RBD)')
ax.set_xlabel('SARS-CoV Position (bp)')
ax.set_ylabel('SARS-CoV-2 Position (bp)')
plt.colorbar(im, ax=ax, label='Score')
plt.tight_layout()
plt.show()